<a href="https://colab.research.google.com/github/syauqidamario/thesis-MTI-syauqi/blob/main/02_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install transformers torch
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from torch.nn.functional import softmax
from tqdm import tqdm

In [2]:
# menampilkan dataset
df = pd.read_csv('01_berita_bbca_prepared.csv')
df.head(5)

,date,aggregated_news,news_count
0,2020-01-01,sahamsaham yang paling untung dan buntung sepa...,1
1,2020-01-15,harga saham bca cetak rekor baru,1
2,2020-01-20,top saham bbri catat rekor tertinggi di tahun ...,1
3,2020-02-12,tps food sajikan ulang lapkeu 2017 rugi memben...,1
4,2020-02-14,tiga anak krakatau steel bersiap ipo bergerak ...,1


In [3]:
# IndoBERT untuk klasifikasi sentimen
model_name = "indobenchmark/indobert-base-p1"
tokenizer = BertTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
# Di sini kita menggunakan model yang sudah di-fine-tune untuk sentimen jika tersedia,
# atau melakukan ekstraksi embedding
model = BertForSequenceClassification.from_pretrained("indobenchmark/indobert-base-p1", num_labels=3) # 3: Neg, Neut, Pos
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [5]:
# ekstraksi score sentimen
def get_sentiment_score(text):
    # Batasi panjang teks sesuai limit BERT (512 token)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = softmax(outputs.logits, dim=1)

    # Menghitung skor komposit: (Prob_Positif - Prob_Negatif)
    # Hasil berada di rentang -1 hingga 1
    neg, neut, pos = probs[0].tolist()
    sentiment_score = pos - neg
    return sentiment_score

# Menjalankan proses skoring pada seluruh baris berita
tqdm.pandas()
df['sentiment_score'] = df['aggregated_news'].progress_apply(get_sentiment_score)


100%|██████████| 675/675 [02:00<00:00,  5.62it/s]


In [6]:
# Dataset siap training
# Simpan dataset untuk Notebook 03: Pelatihan Model Hybrid
df.to_csv('02_dataset_siap_training.csv', index=False)